In [3]:
import numpy as np
import torch
from torchvision.ops.boxes import box_area
def calculate_iou(box1, box2):
    # 计算交集
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    
    # 计算并集
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    if (intersection == 0): 
        return 0
    union = box1_area + box2_area - intersection
    
    # 计算IoU
    iou = intersection / union

    return iou


# # 示例预测边界框和真实''边界框
# pred_boxes = [
#     [50, 50, 150, 150],
#     [30, 30, 120, 120],
#     [200, 200, 300, 300]
# ]

# true_boxes = [
#     [60, 60, 140, 140],
#     [20, 20, 130, 180],
#     [210, 210, 290, 290]
# ]
# pred_boxes, true_boxes = [], []


# modified from torchvision to also return the union
def box_iou(boxes1, boxes2):
    area1 = box_area(boxes1)
    area2 = box_area(boxes2)

    lt = torch.max(boxes1[:, None, :2], boxes2[:, :2])  # [N,M,2]
    rb = torch.min(boxes1[:, None, 2:], boxes2[:, 2:])  # [N,M,2]

    wh = (rb - lt).clamp(min=0)  # [N,M,2]
    inter = wh[:, :, 0] * wh[:, :, 1]  # [N,M]

    union = area1[:, None] + area2 - inter

    iou = inter / union
    return iou, union
    
def generalized_box_iou(boxes1, boxes2):
    """
    Generalized IoU from https://giou.stanford.edu/

    The boxes should be in [x0, y0, x1, y1] format

    Returns a [N, M] pairwise matrix, where N = len(boxes1)
    and M = len(boxes2)
    """
    # degenerate boxes gives inf / nan results
    # so do an early check
    # print(boxes1, boxes2)
    boxes1 = torch.tensor(boxes1)
    boxes2 = torch.tensor(boxes2)
    
    # assert (boxes1[:, 2:] >= boxes1[:, :2]).all()
    # assert (boxes2[:, 2:] >= boxes2[:, :2]).all()
    iou, union = box_iou(boxes1, boxes2)

    lt = torch.min(boxes1[:, None, :2], boxes2[:, :2])
    rb = torch.max(boxes1[:, None, 2:], boxes2[:, 2:])

    wh = (rb - lt).clamp(min=0)  # [N,M,2]
    area = wh[:, :, 0] * wh[:, :, 1]

    return iou - (area - union) / area

def calculate_accuracy(pred_boxes, true_boxes, iou_threshold=0.5):
    correct_predictions = 0
    total_predictions = len(pred_boxes)
    
    for pred_box, true_box in zip(pred_boxes, true_boxes):
        iou = calculate_iou(pred_box, true_box)
        if iou >= iou_threshold:
            correct_predictions += 1
    accuracy = correct_predictions / total_predictions
    return accuracy


        # anno_path = '/mnt/petrelfs/share_data/yanziang/video_agent_data/additional/REC_refcoco+_unc_val.json',
        # anno_path = '/mnt/petrelfs/share_data/yanziang/video_agent_data/additional/REC_refcoco_unc_val.json',
        # anno_path = '/mnt/petrelfs/share_data/yanziang/video_agent_data/additional/REC_refcocog_umd_test.json',
        # anno_path = '/mnt/petrelfs/share_data/yanziang/video_agent_data/additional/REC_refcoco_unc_testB.json',
        # anno_path = '/mnt/petrelfs/share_data/yanziang/video_agent_data/additional/REC_refcoco_unc_testA.json',
        # anno_path = '/mnt/petrelfs/share_data/yanziang/video_agent_data/additional/REC_refcoco+_unc_testB.json',
        # anno_path = '/mnt/petrelfs/share_data/yanziang/video_agent_data/additional/REC_refcoco+_unc_testA.json',
        # anno_path = '/mnt/petrelfs/share_data/yanziang/video_agent_data/additional/REC_refcocog_umd_val.json',
      

import jsonlines
for num in range(8):
    pred_boxes = []
    true_boxes = []
    file = f'/mnt/petrelfs/yanziang/trackingllm/videoagent/logs/results/merge_new/visual_grounding/{num}.jsonl'
    # file = '/mnt/petrelfs/yanziang/trackingllm/NExT-Chat/res3.jsonl'
    with jsonlines.open(file) as f:
        for dd in f:
            # print(dd['pred'], dd['gt'])
            pred_boxes.append(dd['pred'])
            true_boxes.append(dd['gt'][0])
            # pred_boxes.append(dd['ret'][0])
            # true_boxes.append(dd['bbox'])
    # print(len(pred_boxes))
    # print(pred_boxes[10000])
# 计算 acc @ 0.3
    accuracy = calculate_accuracy(pred_boxes, true_boxes, iou_threshold=0.5)
    print(f"Accuracy @ 0.5 IoU: {accuracy:.5f}")

#     Accuracy @ 0.5 IoU: 0.75045
# Accuracy @ 0.5 IoU: 0.83051
# Accuracy @ 0.5 IoU: 0.76292
# Accuracy @ 0.5 IoU: 0.77920
# Accuracy @ 0.5 IoU: 0.87502
# Accuracy @ 0.5 IoU: 0.66946
# Accuracy @ 0.5 IoU: 0.81855
# Accuracy @ 0.5 IoU: 0.75919

# Accuracy @ 0.5 IoU: 0.79420
# Accuracy @ 0.5 IoU: 0.85877
# Accuracy @ 0.5 IoU: 0.80234
# Accuracy @ 0.5 IoU: 0.81335
# Accuracy @ 0.5 IoU: 0.90790
# Accuracy @ 0.5 IoU: 0.71732
# Accuracy @ 0.5 IoU: 0.85121
# Accuracy @ 0.5 IoU: 0.80188

Accuracy @ 0.5 IoU: 0.79420
Accuracy @ 0.5 IoU: 0.85877
Accuracy @ 0.5 IoU: 0.80234
Accuracy @ 0.5 IoU: 0.81335
Accuracy @ 0.5 IoU: 0.90790
Accuracy @ 0.5 IoU: 0.71732
Accuracy @ 0.5 IoU: 0.85121
Accuracy @ 0.5 IoU: 0.80188


In [ ]:
# REC_refcoco+_unc_val
# REC_refcoco_unc_val
# REC_refcocog_umd_test
# REC_refcoco_unc_testB
# REC_refcoco_unc_testA
# REC_refcoco+_unc_testB
# REC_refcoco+_unc_testA
# REC_refcocog_umd_val

In [1]:
import json
from statistics import mean

data_root = "/mnt/petrelfs/yanziang/trackingllm/videoagent/logs/results/merge_new/grounding/"
R1_03 = []
R1_05 = []
R1_07 = []
mIoU = []
HL_mAP = []
HL_Hit1 = []

for i in range(8):
    # with open(data_root+f'{i}-hl_metrics.json', 'r') as f:
    with open(data_root+f'{i}-c_metrics.json', 'r') as f:
        data = json.load(f)
        R1_03.append(data["brief"]["MR-full-R1@0.3"])
        R1_05.append(data["brief"]["MR-full-R1@0.5"])
        R1_07.append(data["brief"]["MR-full-R1@0.7"])
        mIoU.append(data["brief"]["MR-full-mIoU"])

        if "HL-min-VeryGood-mAP" in data["brief"]:
            HL_mAP.append(data["brief"]["HL-min-VeryGood-mAP"])
            HL_Hit1.append(data["brief"]["HL-min-VeryGood-Hit1"])

print("R1@0.3: ", mean(R1_03))
print("R1@0.5: ", mean(R1_05))
print("R1@0.7: ", mean(R1_07))
print("mIoU: ", mean(mIoU))
if "HL-min-VeryGood-mAP" in data["brief"]:
    print("HL_mAP: ", mean(HL_mAP))
    print("HL_Hit1: ", mean(HL_Hit1))

# R1@0.3:  58.27875
# R1@0.5:  40.21375
# R1@0.7:  18.4125

# HL_mAP:  35.66375
# HL_Hit1:  62.394999999999996

R1@0.3:  55.879999999999995
R1@0.5:  34.60125
R1@0.7:  15.165
mIoU:  35.8425
